<a href="https://colab.research.google.com/github/josenomberto/UTEC-CDIAV3-MCD8016/blob/main/proyecto1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Configuración del Entorno y Preparación de Datos (Eficiencia SSD)
Esta celda se encarga de montar Google Drive, copiar los archivos comprimidos pesados al almacenamiento local temporal rápido de Colab y extraerlos de manera silenciosa para optimizar las operaciones de entrada/salida (I/O).


In [16]:

from google.colab import drive
import os

# 1. Montar Google Drive
drive.mount('/content/drive')

# Definir rutas de origen en tu Drive (Ajusta las rutas según tu estructura)
DRIVE_TRAIN_ZIP = "/content/drive/MyDrive/MCD8016-DeepLearning/Proyecto1/Animal Sounds/train.7z"
DRIVE_TEST_ZIP = "/content/drive/MyDrive/MCD8016-DeepLearning/Proyecto1/Animal Sounds/test.7z"
DRIVE_CSV = "/content/drive/MyDrive/MCD8016-DeepLearning/Proyecto1/Animal Sounds/train.csv"

# 2. Copiar al almacenamiento SSD local temporal de Colab (/content/)
print("Copiando archivos desde el Drive compartido al disco local...")
if os.path.exists(DRIVE_TRAIN_ZIP):
    !cp "{DRIVE_TRAIN_ZIP}" /content/train.7z
    print("¡Archivo train.7z copiado exitosamente!")
else:
    print(f"ADVERTENCIA: No se encontró acceso directo para train.7z en {DRIVE_TRAIN_ZIP}")

if os.path.exists(DRIVE_TEST_ZIP):
    !cp "{DRIVE_TEST_ZIP}" /content/test.7z
    print("¡Archivo test.7z copiado exitosamente!")

if os.path.exists(DRIVE_CSV):
    !cp "{DRIVE_CSV}" /content/train.csv
    print("¡Archivo train.csv copiado exitosamente!")


# 3. Extracción local rápida usando 7z
print("Extrayendo conjunto de entrenamiento...")
if os.path.exists("/content/train.7z"):
    !7z x /content/train.7z -o/content/train_data -bd > /dev/null
    print("¡Extracción de entrenamiento completada!")
else:
    print("ADVERTENCIA: No se encontró train.7z en el directorio local.")

print("Extrayendo conjunto de prueba...")
if os.path.exists("/content/test.7z"):
    !7z x /content/test.7z -o/content/test_data -bd > /dev/null
    print("¡Extracción de prueba completada!")
else:
    print("ADVERTENCIA: No se encontró test.7z en el directorio local.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copiando archivos desde el Drive compartido al disco local...
¡Archivo train.7z copiado exitosamente!
¡Archivo test.7z copiado exitosamente!
¡Archivo train.csv copiado exitosamente!
Extrayendo conjunto de entrenamiento...
¡Extracción de entrenamiento completada!
Extrayendo conjunto de prueba...
¡Extracción de prueba completada!


## 2. Protocolo de Validación y Cargador de Datos Perezoso (*Lazy Loading*)
Establecemos la semilla de aleatoriedad global requerida para asegurar la reproducibilidad. Implementamos el cargador `AmazonWildlifeDataset` utilizando **Lazy Loading** (carga bajo demanda) y la extracción de espectrogramas Mel en GPU para prevenir problemas de falta de memoria RAM.


In [4]:
import torch
import torchaudio
import pandas as pd
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# --- 1. CONFIGURACIÓN DE SEMILLAS PARA REPRODUCIBILIDAD ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print(f"Semilla aleatoria global establecida en {SEED}")

# --- 2. DIVISIÓN DEL DATASET (80% TRAIN / 20% VAL) ---
try:
    df_completo = pd.read_csv('/content/drive/MyDrive/MCD8016-DeepLearning/Proyecto1/Animal Sounds/train.csv')
    df_train, df_val = train_test_split(df_completo, test_size=0.20, random_state=SEED)
    print(f"Dataset cargado. Muestras entrenamiento: {len(df_train)} | Muestras validación: {len(df_val)}")
except Exception as e:
    print("Error al cargar train.csv. Asegúrate de que el archivo esté en /content/train.csv")
    print(e)

# --- 3. DEFINICIÓN DEL DATASET CON LAZY LOADING ---
class AmazonWildlifeDataset(Dataset):
    def __init__(self, df, audio_dir, target_sr=22050, duration=3, transform=None):
        self.df = df
        self.audio_dir = audio_dir
        self.target_sr = target_sr
        self.num_samples = target_sr * duration
        self.transform = transform

        # El nombre del archivo está en la columna 0, las 42 especies en las restantes
        self.filenames = self.df.iloc[:, 0].values
        self.labels = self.df.iloc[:, 1:].values.astype('float32')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        audio_path = os.path.join(self.audio_dir, self.filenames[idx])

        try:
            waveform, sr = torchaudio.load(audio_path)

            # Conversión a mono si tuviera más canales
            if waveform.shape > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            # Resampling si la tasa difiere
            if sr != self.target_sr:
                resampler = torchaudio.transforms.Resample(sr, self.target_sr)
                waveform = resampler(waveform)

            # Padding o Truncamiento a 3 segundos exactos para mantener dimensiones consistentes
            if waveform.shape > self.num_samples:
                waveform = waveform[:, :self.num_samples]
            elif waveform.shape < self.num_samples:
                pad_len = self.num_samples - waveform.shape
                waveform = torch.nn.functional.pad(waveform, (0, pad_len))
        except Exception:
            # Fallback en caso de archivo inaccesible (vector de ceros)
            waveform = torch.zeros((1, self.num_samples))

        if self.transform:
            waveform = self.transform(waveform)

        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return waveform, label

Semilla aleatoria global establecida en 42
Dataset cargado. Muestras entrenamiento: 49752 | Muestras validación: 12439


## 3. Extractor de Características Espectrales en GPU
Definimos el extractor utilizando espectrogramas de Mel de alta resolución e inicializamos los cargadores de PyTorch configurados con hilos de CPU en paralelo (`num_workers=2`) para acelerar la transferencia de datos.


In [5]:
# --- 4. EXTRACTOR DE ESPECTROGRAMA DE MEL EN GPU ---
# Usamos un MelSpectrogram optimizado de alta resolución (N_fft=2048, n_mels=128)
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=22050,
    n_fft=2048,
    hop_length=512,
    n_mels=128
)

# Instanciar Datasets locales
train_dataset = AmazonWildlifeDataset(df_train, '/content/train_data', transform=mel_transform)
val_dataset = AmazonWildlifeDataset(df_val, '/content/train_data', transform=mel_transform)

# Dataloaders optimizados con multiprocesamiento
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Verificación rápida de dimensiones
for x, y in train_loader:
    print(f"Tensor de espectrogramas Mel (Batch, Canales, Mels, Pasos temporales): {x.shape}")
    print(f"Tensor de etiquetas multi-label (Batch, Clases): {y.shape}")
    break


Tensor de espectrogramas Mel (Batch, Canales, Mels, Pasos temporales): torch.Size([32, 1, 128, 130])
Tensor de etiquetas multi-label (Batch, Clases): torch.Size([32, 42])


## 4. Implementación Propia de los Módulos de DTFP (PyTorch)
Codificamos de forma nativa los componentes propuestos en el artículo: `DB_Conv`, `CSTF_Enhancement` con el mecanismo de desplazamiento circular de canales (*Cross-Shifting*), el predictor de ventanas adaptativas `CWP` y el bloque de atención local de doble escala `MLDSA`.


In [11]:
import torch.nn as nn
import torch.nn.functional as F

# --- Bloque A: Dual-Branch Convolution (DB_Conv) ---
class DB_Conv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid_channels = out_channels // 2 if out_channels >= 2 else 1

        # Rama temporal (1x3)
        self.conv_time_1x3 = nn.Conv2d(in_channels, mid_channels, kernel_size=(1,3), stride=(2,1), padding=(0,1))
        self.conv_time_3x3 = nn.Conv2d(mid_channels, out_channels, kernel_size=(3,3), stride=(1,1), padding=(1,1))

        # Rama frecuencial (3x1)
        self.conv_freq_3x1 = nn.Conv2d(in_channels, mid_channels, kernel_size=(3,1), stride=(2,1), padding=(1,0))
        self.conv_freq_3x3 = nn.Conv2d(mid_channels, out_channels, kernel_size=(3,3), stride=(1,1), padding=(1,1))

        # Conexión residual 1x1 con downsampling de frecuencia
        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(2,1))

        self.bn = nn.BatchNorm2d(out_channels)
        self.avg_pool = nn.AvgPool2d(kernel_size=(2,1), stride=(2,1)) # Frecuencia / 2 adicional
        self.relu = nn.ReLU()

    def forward(self, x):
        # Entrada: B x C x F x T
        # Rama temporal
        xt = self.conv_time_1x3(x)
        xt = self.conv_time_3x3(xt)

        # Rama frecuencial
        xf = self.conv_freq_3x1(x)
        xf = self.conv_freq_3x3(xf)

        # Fusión residual
        res = self.residual_conv(x)
        out = self.relu(self.bn(xt + xf) + res)

        # Promedio del eje de frecuencia
        out = self.avg_pool(out)
        return out

# --- Bloque B: Bloque CSTF Enhancement con Cross-Shifting circular ---
class CSTF_Enhancement(nn.Module):
    def __init__(self, channels, num_groups=4, enable_shift=True):
        super().__init__()
        self.channels = channels
        self.num_groups = num_groups
        self.enable_shift = enable_shift

        # Kernels dinámicos para modelado de frecuencia (FDC) y tiempo (TDC)
        self.conv_fdc = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=num_groups)
        self.conv_tdc = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=num_groups)

        self.bn1 = nn.BatchNorm2d(channels)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()

    def cross_shift(self, x, axis='f'):
        if not self.enable_shift:
            return x
        B, C, F, T = x.shape
        group_size = C // self.num_groups
        shifted_groups = []

        for i in range(self.num_groups):
            group = x[:, i*group_size:(i+1)*group_size]
            shift = 1 if i % 2 == 0 else -1

            if axis == 'f':
                shifted = torch.roll(group, shifts=shift, dims=2) # Desplazamiento circular en eje frecuencia
            else:
                shifted = torch.roll(group, shifts=shift, dims=3) # Desplazamiento circular en eje tiempo
            shifted_groups.append(shifted)

        return torch.cat(shifted_groups, dim=1)

    def forward(self, x):
        # Rama 1: FDC + Cross Shifting Frecuencial
        x1 = self.relu(self.bn1(self.conv_fdc(x)))
        x1 = self.cross_shift(x1, axis='f')

        # Rama 2: Identidad residual
        x2 = x

        # Rama 3: TDC + Cross Shifting Temporal
        x3 = self.relu(self.bn2(self.conv_tdc(x)))
        x3 = self.cross_shift(x3, axis='t')

        # Fusión mediante suma circular de canales
        return x1 + x2 + x3

# --- Bloque C: Predictor de Ventana Contextual (CWP) ---
class ContextWindowPrediction(nn.Module):
    def __init__(self, in_features, min_w=15, max_w=51):
        super().__init__()
        self.min_w = min_w
        self.max_w = max_w
        self.fc = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, g_prime):
        # g_prime: B x T x D
        # Calcule la representación global promediando a lo largo del tiempo para obtener B x D
        global_representation = torch.mean(g_prime, dim=1) # B x D

        # Inferencia del factor de complejidad Gamma
        gamma = self.fc(global_representation) # B x 1

        # Asignación dinámica de la ventana
        N_b = self.min_w + (1.0 - gamma) * (self.max_w - self.min_w)
        N_l = torch.clamp(N_b, self.min_w, self.max_w).mean().int().item()
        N_s = max(self.min_w, int(N_l * 0.6)) # Ventana pequeña proporcional

        # Asegurar valores impares para la vecindad de atención
        if N_l % 2 == 0: N_l += 1
        if N_s % 2 == 0: N_s += 1
        return N_l, N_s

# --- Bloque D: Multi-scale Local Dense Synthesizer Attention (MLDSA) ---
class MLDSA_Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

        self.proj_l1 = nn.Linear(dim, dim // 4)
        self.proj_l2 = nn.Linear(dim // 4, dim)
        self.proj_s1 = nn.Linear(dim, dim // 4)
        self.proj_s2 = nn.Linear(dim // 4, dim)

        self.value_proj = nn.Linear(dim, dim)
        self.fusion_weight = nn.Sequential(
            nn.Linear(dim, 1),
            nn.Sigmoid()
        )
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x, N_l, N_s):
        # x: B x T x D
        B, T, D = x.shape
        V = self.value_proj(x)

        # Rama de Ventana Grande (N_l)
        attn_l = F.softmax(self.proj_l2(F.relu(self.proj_l1(x))), dim=-1) # B x T x D
        # Rama de Ventana Chica (N_s)
        attn_s = F.softmax(self.proj_s2(F.relu(self.proj_s1(x))), dim=-1) # B x T x D

        # Agregación local ponderada por vecindad temporal
        Y_l = torch.zeros_like(V)
        Y_s = torch.zeros_like(V)

        for t in range(T):
            # Ventana Grande
            l_start = max(0, t - N_l // 2)
            l_end = min(T, t + N_l // 2 + 1)
            Y_l[:, t, :] = torch.mean(attn_l[:, t, None] * V[:, l_start:l_end, :], dim=1)

            # Ventana Chica
            s_start = max(0, t - N_s // 2)
            s_end = min(T, t + N_s // 2 + 1)
            Y_s[:, t, :] = torch.mean(attn_s[:, t, None] * V[:, s_start:s_end, :], dim=1)

        # Fusión adaptativa por trama secuencial
        xi = self.fusion_weight(x) # B x T x 1
        Y_fused = xi * Y_s + (1.0 - xi) * Y_l
        return self.out_proj(Y_fused)

## 5. Arquitectura de Red Unificada
Definimos la red unificada con el extractor Mel, los bloques CSTF y MLDSA, y un modelado secuencial con **BiGRU**. Integramos la adaptación a nivel de clip colapsando la dimensión temporal mediante **Global Average Pooling** para proyectar logits hacia las **42 clases**.


In [7]:
# --- Bloque E: Arquitectura Completa Adaptada a la Amazonia ---
class DTFP_AmazoniaClassifier(nn.Module):
    def __init__(self, num_classes=42, enable_shift=True):
        super().__init__()
        # Extracción y reducción convolucional
        self.db_conv1 = DB_Conv(in_channels=1, out_channels=16)     # F -> F/4 (128 -> 32)
        self.cstf = CSTF_Enhancement(channels=16, enable_shift=enable_shift)
        self.db_conv2 = DB_Conv(in_channels=16, out_channels=32)    # F/4 -> F/16 (32 -> 8)

        # Adaptación dimensional para MLDSA: Canales x Frecuencia_final = 32 * 8 = 256
        self.cwp = ContextWindowPrediction(in_features=256)
        self.mldsa = MLDSA_Attention(dim=256)

        # Modelado secuencial bidireccional
        self.bigru = nn.GRU(
            input_size=256,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        # Clasificador multietiqueta a nivel de clip completo
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(256, num_classes) # BiGRU produce 2 * 128 = 256 características
        )

    def forward(self, x):
        # Entrada: B x 1 x F x T
        x = self.db_conv1(x)
        x = self.cstf(x)
        x = self.db_conv2(x) # Salida: B x 32 x 8 x T

        # Reshape espacial-temporal
        B, C, F, T = x.shape
        x_reshaped = x.permute(0, 3, 1, 2).contiguous() # B x T x C x F
        x_reshaped = x_reshaped.view(B, T, C * F)       # B x T x 256

        # Inferencia de ventanas en CWP y atención MLDSA
        N_l, N_s = self.cwp(x_reshaped)
        x_attn = self.mldsa(x_reshaped, N_l, N_s)

        # GRU Bidireccional
        h_seq, _ = self.bigru(x_attn) # B x T x 256

        # Adaptación: Global Average Pooling sobre el eje del tiempo para clip completo
        h_clip = torch.mean(h_seq, dim=1) # B x 256

        logits = self.classifier(h_clip) # B x 42
        return logits

## 6. Bucle de Entrenamiento, Validación y Métricas de Rendimiento
Esta celda implementa las funciones de evaluación computando las métricas requeridas: **mAP** (curva Precision-Recall macro), **F1-Macro**, **F1-Micro** y la exigente **Subset Accuracy** (Exactitud de subconjunto).


In [12]:
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, accuracy_score
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

def evaluate_model(model, dataloader, device, threshold=0.5):
    model.eval()
    all_targets = []
    all_outputs = []
    val_loss = 0.0
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for waveforms, labels in dataloader:
            waveforms = waveforms.to(device)
            labels = labels.to(device)

            with autocast():
                logits = model(waveforms)
                loss = criterion(logits, labels)

            val_loss += loss.item() * waveforms.size(0)
            probs = torch.sigmoid(logits)

            all_targets.append(labels.cpu().numpy())
            all_outputs.append(probs.cpu().numpy())

    val_loss /= len(dataloader.dataset)
    all_targets = np.vstack(all_targets)
    all_outputs = np.vstack(all_outputs)

    # Cálculo de Métricas Multietiqueta
    mAP = average_precision_score(all_targets, all_outputs, average="macro")
    preds = (all_outputs >= threshold).astype(int)

    # Exactitud de subconjunto (Exacta coincidencia en las 42 etiquetas)
    subset_acc = accuracy_score(all_targets, preds)

    # Micro y Macro F1
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(all_targets, preds, average="macro", zero_division=0)
    p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(all_targets, preds, average="micro", zero_division=0)

    return val_loss, mAP, f1_macro, f1_micro, subset_acc

# Bucle Principal de Entrenamiento (Modelo Completo)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Entrenando sobre dispositivo: {device}")

model = DTFP_AmazoniaClassifier(num_classes=42, enable_shift=True).to(device)
criterion = nn.BCEWithLogitsLoss() # Pérdida a nivel de clip completo
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=4e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
scaler = GradScaler()

num_epochs = 10 # Ajustar según tu tiempo disponible en Colab
best_val_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for waveforms, labels in train_loader:
        waveforms = waveforms.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        with autocast():
            logits = model(waveforms)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * waveforms.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    scheduler.step()

    # Evaluar en el conjunto del 20% de validación
    val_loss, mAP, f1_macro, f1_micro, subset_acc = evaluate_model(model, val_loader, device)

    print(f"Época [{epoch+1}/{num_epochs}] -> Pérdida Train: {train_loss:.4f} | Pérdida Val: {val_loss:.4f}")
    print(f"   Validación: mAP={mAP:.4f} | F1-Macro={f1_macro:.4f} | F1-Micro={f1_micro:.4f} | Exactitud Subconjunto={subset_acc:.4f}\n")

    # Guardar mejor modelo basado en validación
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_dtfp_amazonia_model.pth')
        print("   ¡Modelo óptimo guardado con éxito!")


Entrenando sobre dispositivo: cuda


/tmp/ipykernel_1246/1259631698.py:52: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1246/1259631698.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1246/1259631698.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Época [1/10] -> Pérdida Train: 0.1400 | Pérdida Val: 0.1280
   Validación: mAP=0.0359 | F1-Macro=0.0000 | F1-Micro=0.0000 | Exactitud Subconjunto=0.3644

   ¡Modelo óptimo guardado con éxito!


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/tmp/ipykernel_1246/1259631698.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1246/1259631698.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Época [2/10] -> Pérdida Train: 0.1293 | Pérdida Val: 0.1281
   Validación: mAP=0.0359 | F1-Macro=0.0000 | F1-Micro=0.0000 | Exactitud Subconjunto=0.3644



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/tmp/ipykernel_1246/1259631698.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


KeyboardInterrupt: 

## 7. Experimento de Ablación Obligatorio (Bloque CSTF)
Desactivamos el mecanismo de desplazamiento cruzado circular de canales (`enable_shift=False`) en nuestro módulo de espectros `CSTF` y entrenamos un modelo con la misma semilla.


In [ ]:
# Entrenamiento del Modelo de Ablación (Sin Cross-Shifting)
print("Inicializando modelo de ablación (enable_shift=False)...\n")
model_ablation = DTFP_AmazoniaClassifier(num_classes=42, enable_shift=False).to(device)

optimizer_abl = optim.AdamW(model_ablation.parameters(), lr=2e-4, weight_decay=4e-5)
scheduler_abl = optim.lr_scheduler.CosineAnnealingLR(optimizer_abl, T_max=10, eta_min=1e-6)
scaler_abl = GradScaler()
best_val_loss_abl = float('inf')

for epoch in range(num_epochs):
    model_ablation.train()
    running_loss = 0.0

    for waveforms, labels in train_loader:
        waveforms = waveforms.to(device)
        labels = labels.to(device)

        optimizer_abl.zero_grad()
        with autocast():
            logits = model_ablation(waveforms)
            loss = criterion(logits, labels)

        scaler_abl.scale(loss).backward()
        scaler_abl.step(optimizer_abl)
        scaler_abl.update()

        running_loss += loss.item() * waveforms.size(0)

    train_loss_abl = running_loss / len(train_loader.dataset)
    scheduler_abl.step()

    val_loss_abl, mAP_abl, f1_macro_abl, f1_micro_abl, subset_acc_abl = evaluate_model(model_ablation, val_loader, device)

    print(f"Época Ablación [{epoch+1}/{num_epochs}] -> Pérdida Train: {train_loss_abl:.4f} | Pérdida Val: {val_loss_abl:.4f}")
    print(f"   Ablación Val: mAP={mAP_abl:.4f} | F1-Macro={f1_macro_abl:.4f} | F1-Micro={f1_micro_abl:.4f} | Exactitud Subconjunto={subset_acc_abl:.4f}\n")

    if val_loss_abl < best_val_loss_abl:
        best_val_loss_abl = val_loss_abl
        torch.save(model_ablation.state_dict(), 'best_dtfp_ablation_model.pth')
        print("   ¡Modelo ablacionado óptimo guardado!")

# Tabla Comparativa Resumen del Experimento de Ablación
print("Evaluando modelos finales...")
# Cargar el modelo base y obtener sus métricas
model.load_state_dict(torch.load('best_dtfp_amazonia_model.pth'))
val_loss, mAP, f1_macro, f1_micro, subset_acc = evaluate_model(model, val_loader, device)

# Cargar el modelo de ablación y obtener sus métricas
model_ablation.load_state_dict(torch.load('best_dtfp_ablation_model.pth'))
val_loss_abl, mAP_abl, f1_macro_abl, f1_micro_abl, subset_acc_abl = evaluate_model(model_ablation, val_loader, device)

tabla_resultados = pd.DataFrame({
    "Métrica": ["Pérdida Validación", "Mean Average Precision (mAP)", "F1-Score Macro", "F1-Score Micro", "Exactitud de Subconjunto"],
    "DTFP Completo (Propuesta)": [val_loss, mAP, f1_macro, f1_micro, subset_acc],
    "DTFP sin Cross-Shifting (Ablación)": [val_loss_abl, mAP_abl, f1_macro_abl, f1_micro_abl, subset_acc_abl]
})

print("\n--- TABLA DE COMPARATIVA FINAL DEL EXPERIMENTO DE ABLACIÓN ---")
display(tabla_resultados)

## 8. Inferencia del Conjunto de Prueba
Utilizamos el mejor modelo entrenado para inferir sobre las grabaciones sin etiquetas de la carpeta `test/` y estructurar el archivo de salida formateado con el nombre de archivo y la presencia/ausencia binaria (0 y 1) de las 42 especies.


In [ ]:

import glob

# Obtener nombres de los archivos en test_data
test_files = glob.glob("/content/test_data/*.wav")
test_filenames = [os.path.basename(f) for f in test_files]
print(f"Se encontraron {len(test_filenames)} archivos en el conjunto de prueba test/")

if len(test_filenames) > 0:
    df_test_dummy = pd.DataFrame({
        "file_name": test_filenames
    })
    # Rellenar con 42 columnas dummy de ceros para cumplir la firma del Dataset
    for i in range(42):
        df_test_dummy[f"class_{i}"] = 0

    test_dataset = AmazonWildlifeDataset(df_test_dummy, '/content/test_data', transform=mel_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

    model.eval()
    test_preds = []
    UMBRAL_DECISION = 0.5

    with torch.no_grad():
        for waveforms, _ in test_loader:
            waveforms = waveforms.to(device)
            with autocast():
                logits = model(waveforms)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds_bin = (probs >= UBRAL_DECISION).astype(int)
            test_preds.append(preds_bin)

    test_preds = np.vstack(test_preds)

    # Crear DataFrame de entrega final con los nombres reales de las 42 especies
    columnas_especies = df_completo.columns[1:] if 'df_completo' in locals() else [f"species_{i}" for i in range(42)]
    df_submission = pd.DataFrame(test_preds, columns=columnas_especies)
    df_submission.insert(0, 'file_name', test_filenames)

    # Guardar CSV
    df_submission.to_csv('submission_results.csv', index=False)
    print("¡Archivo 'submission_results.csv' generado con éxito para su entrega!")
else:
    print("No se encontraron archivos en test_data para realizar la inferencia.")
